In [2]:
import requests

In [3]:
import re
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

In [3]:
htmls = []
cnt = 0
missed = []

In [86]:
for i in range(1,50):
    r = requests.get(f'https://www.flipkart.com/search?q=laptop&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page={i}')
    html = r.text
    htmls.append(html)
    if html:
        print("succesfull got data from page ",i)
        cnt+=1
    else:
        missed.append(i)
        

succesfull got data from page  1
succesfull got data from page  2
succesfull got data from page  3
succesfull got data from page  4
succesfull got data from page  5
succesfull got data from page  7
succesfull got data from page  8
succesfull got data from page  9
succesfull got data from page  10
succesfull got data from page  11
succesfull got data from page  12
succesfull got data from page  13
succesfull got data from page  14
succesfull got data from page  15
succesfull got data from page  16
succesfull got data from page  17
succesfull got data from page  18
succesfull got data from page  19
succesfull got data from page  20
succesfull got data from page  21
succesfull got data from page  22
succesfull got data from page  23
succesfull got data from page  24
succesfull got data from page  25
succesfull got data from page  26
succesfull got data from page  27
succesfull got data from page  28
succesfull got data from page  29
succesfull got data from page  30
succesfull got data fr

In [89]:
import json

In [34]:
with open("data.json", "w") as f:
    json.dump(htmls, f)

In [92]:
len(htmls)

198

In [93]:
for i in range(50,100):
    r = requests.get(f'https://www.flipkart.com/search?q=laptop&sid=6bo%2Cb5g&as=on&as-show=on&otracker=AS_QueryStore_HistoryAutoSuggest_1_6_na_na_na&otracker1=AS_QueryStore_HistoryAutoSuggest_1_6_na_na_na&as-pos=1&as-type=HISTORY&suggestionId=laptop%7CLaptops&requestId=fd7a8980-e1e8-4411-affd-7d89a6ac9b4f&as-searchtext=laptop&page={i}')
    html = r.text
    htmls.append(html)
    if html:
        print("succesfull got data from page ",i)
        cnt+=1
    else:
        missed.append(i)

succesfull got data from page  50
succesfull got data from page  51
succesfull got data from page  52
succesfull got data from page  53
succesfull got data from page  54
succesfull got data from page  55
succesfull got data from page  56
succesfull got data from page  57
succesfull got data from page  58
succesfull got data from page  59
succesfull got data from page  60
succesfull got data from page  61
succesfull got data from page  62
succesfull got data from page  63
succesfull got data from page  64
succesfull got data from page  65
succesfull got data from page  66
succesfull got data from page  67
succesfull got data from page  68
succesfull got data from page  69
succesfull got data from page  70
succesfull got data from page  71
succesfull got data from page  72
succesfull got data from page  73
succesfull got data from page  74
succesfull got data from page  75
succesfull got data from page  76
succesfull got data from page  77
succesfull got data from page  78
succesfull got

In [256]:
import re
from bs4 import BeautifulSoup

# Matches rtx/gtx/mx + 3-4 digits + optional ti/super/max-q using spaces OR hyphens
GPU_NVIDIA = re.compile(r'\b(rtx|gtx|mx)[\s\-]*(\d{3,4})(?:[\s\-]*(ti|super))?\b', re.I)
GPU_AMD = re.compile(r'\b(rx)[\s\-]*(\d{3,4})(?:[\s\-]*(m|s|xt|xtx))?\b', re.I)
GPU_ARC = re.compile(r'\barc[\s\-]*(a\d{3}m?|b\d{3}m?)\b', re.I)

def extract_gpu(href, title, processor):
    slug = (href or "").split("/p/")[0].lower()
    t = (title or "").lower()
    combined = f"{slug} {t}"  # Search both URL slug and title at once

    # 1. Check NVIDIA (RTX / GTX / MX)
    m = GPU_NVIDIA.search(combined)
    if m:
        series = m.group(1).upper()
        model = m.group(2)
        suffix = f" {m.group(3).title()}" if m.group(3) else ""
        return f"NVIDIA GeForce {series} {model}{suffix}"

    # 2. Check AMD Dedicated (RX series)
    m = GPU_AMD.search(combined)
    if m:
        model = m.group(2)
        suffix = m.group(3).upper() if m.group(3) else ""
        return f"AMD Radeon RX {model}{suffix}"

    # 3. Check Intel Arc Dedicated (e.g., Arc A370M, A530M)
    m = GPU_ARC.search(combined)
    if m:
        return f"Intel Arc {m.group(1).upper()}"

    # 4. Fallback if title/slug only says "4 GB Graphics" / "4GB Graphics"
    m = re.search(r'(\d+)\s*gb[\s\-]+graphics', combined)
    if m:
        return f"Dedicated GPU ({m.group(1)} GB)"

    # 5. Integrated fallback based on processor
    p = (processor or "").lower()
    if "apple" in p:
        return "Apple Integrated GPU"
    if "snapdragon" in p:
        return "Qualcomm Adreno"
    if "mediatek" in p:
        return "ARM Mali (Integrated)"
    if "ultra" in p:
        return "Intel Arc Graphics (Integrated)"
    if any(x in p for x in ["core 5", "core 7", "core i5", "core i7", "core i9"]):
        # 7th-10th Gen Intel i5/i7/i9 had Intel UHD, not Iris Xe (Iris Xe started in 11th Gen)
        if any(gen in p for gen in ["(7th gen)", "(8th gen)", "(9th gen)", "(10th gen)", "(5th gen)"]):
            return "Intel UHD Graphics (Integrated)"
        return "Intel Iris Xe (Integrated)"
    if "intel" in p:
        return "Intel UHD Graphics (Integrated)"
    if "ryzen" in p or "amd" in p:
        return "AMD Radeon Graphics (Integrated)"
    return "Unknown"

def scrape_flipkart_laptops(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    products = []

    for card in soup.find_all("a", href=re.compile(r"/p/itm")):
        item = {
            "Title": None, "Price": None, "Processor": None, "GPU": None,
            "RAM": None, "Storage": None, "Operating System": None,
            "Display": None, "Warranty": None, "Additional Features": ""
        }

        title_el = card.find("div", class_="RG5Slk")
        if title_el:
            item["Title"] = title_el.get_text(strip=True)

        price_el = card.find(string=re.compile(r"^₹[0-9,]+"))
        if price_el:
            item["Price"] = price_el.strip()

        for li in card.find_all("li"):
            spec = li.get_text(strip=True)
            s = spec.lower()

            if any(x in s for x in ["processor", "ryzen", "core"]):
                item["Processor"] = spec
            elif "ram" in s or "memory" in s:
                item["RAM"] = spec
            elif any(x in s for x in ["ssd", "hdd", "emmc"]):
                item["Storage"] = spec
            elif any(x in s for x in ["operating system", "windows", "chrome", "mac os", "android"]):
                item["Operating System"] = spec
            elif any(x in s for x in ["inch", "display"]):
                item["Display"] = spec
            elif "warranty" in s or "onsite service" in s:
                item["Warranty"] = spec
            else:
                item["Additional Features"] += (" | " if item["Additional Features"] else "") + spec

        if item["Title"]:
            item["GPU"] = extract_gpu(card.get("href"), item["Title"], item["Processor"])
            products.append(item)

    return products

In [257]:
answers=[]

In [258]:
for html in htmls:
    details = scrape_flipkart_laptops(html)
    answers.extend(details)
    

In [259]:
len(answers)

1920

In [260]:
df = pd.DataFrame(answers)

In [7]:
df = pd.read_csv("Dirty.csv")

In [8]:
df[df.isna().any(axis=1)]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [9]:
df = df[df['Processor'].notna()]

In [10]:
df=df.reset_index(drop = True)

In [11]:
df

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
0,0,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹47,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
1,1,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,"₹34,990",AMD Ryzen 3 Processor,AMD Radeon Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Home Operating System,35.56 cm (14 inch) Display
2,2,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,"₹63,990",Intel Core i5 Processor (13th Gen),Intel Iris Xe (Integrated),16 GB LPDDR4X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
3,3,DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...,"₹1,19,990",AMD Ryzen 7 Processor,NVIDIA GeForce RTX 3050,16 GB DDR5 RAM,512 GB SSD,Windows 11 Home Operating System,38.86 cm (15.3 inch) Display
4,4,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,"₹14,990",MediaTek Kompanio 520 Processor,ARM Mali (Integrated),4 GB LPDDR4X RAM,32 GB EMMC Storage,Chrome Operating System,29.46 cm (11.6 Inch) Display
...,...,...,...,...,...,...,...,...,...
1911,1911,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,"₹59,390",Intel Core i3 Processor (13th Gen),Intel UHD Graphics (Integrated),8 GB DDR5 RAM,512 GB SSD,"Windows 11 Home, Microsoft Office 2024",35.56 cm (14 inch) Display
1912,1912,Samsung Galaxy Book4 Edge Series Copilot AI-PC...,"₹63,990",Snapdragon X Processor,Qualcomm Adreno,16 GB LPDDR5X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
1913,1913,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹49,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 inch) Display
1914,1914,Acer Aspire 7 Intel Core i5 13th Gen - (16 GB/...,"₹86,755",Intel Core i5 Processor (13th Gen),NVIDIA GeForce RTX 2050,16 GB DDR4 RAM,512 GB SSD,Windows 11 Operating System,39.62 cm (15.6 inch) Display


In [12]:
df[df.isna().any(axis=1)]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [13]:
dirty = df[df.isna().any(axis=1)]

In [14]:
dirty['Storage'] =  dirty['Title'].str.extract(r'GB/\s*([^/]+)', flags=re.IGNORECASE)


In [15]:
dirty

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [16]:
mapping = dict(zip(dirty['Title'], dirty['Storage']))

In [17]:
mapping

{}

In [18]:
df['Storage'] = df['Storage'].fillna(df['Title'].map(mapping))

In [19]:
dirty = df[df.isna().any(axis=1)]
df = df.dropna()

In [20]:
df = df.reset_index(drop=True)

In [21]:
df

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
0,0,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹47,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
1,1,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,"₹34,990",AMD Ryzen 3 Processor,AMD Radeon Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Home Operating System,35.56 cm (14 inch) Display
2,2,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,"₹63,990",Intel Core i5 Processor (13th Gen),Intel Iris Xe (Integrated),16 GB LPDDR4X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
3,3,DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...,"₹1,19,990",AMD Ryzen 7 Processor,NVIDIA GeForce RTX 3050,16 GB DDR5 RAM,512 GB SSD,Windows 11 Home Operating System,38.86 cm (15.3 inch) Display
4,4,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,"₹14,990",MediaTek Kompanio 520 Processor,ARM Mali (Integrated),4 GB LPDDR4X RAM,32 GB EMMC Storage,Chrome Operating System,29.46 cm (11.6 Inch) Display
...,...,...,...,...,...,...,...,...,...
1911,1911,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,"₹59,390",Intel Core i3 Processor (13th Gen),Intel UHD Graphics (Integrated),8 GB DDR5 RAM,512 GB SSD,"Windows 11 Home, Microsoft Office 2024",35.56 cm (14 inch) Display
1912,1912,Samsung Galaxy Book4 Edge Series Copilot AI-PC...,"₹63,990",Snapdragon X Processor,Qualcomm Adreno,16 GB LPDDR5X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
1913,1913,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹49,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 inch) Display
1914,1914,Acer Aspire 7 Intel Core i5 13th Gen - (16 GB/...,"₹86,755",Intel Core i5 Processor (13th Gen),NVIDIA GeForce RTX 2050,16 GB DDR4 RAM,512 GB SSD,Windows 11 Operating System,39.62 cm (15.6 inch) Display


In [22]:
clean = pd.DataFrame()

In [23]:
clean['Price'] = df['Price'].str.replace(r'\D', '', regex=True).astype(int)

In [24]:
df[df.isna().any(axis=1)]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [25]:
set(df['Operating System'])

{'32 bit Windows 11 Operating System',
 '64 bit Android Operating System',
 '64 bit Chrome Operating System',
 '64 bit DOS Operating System',
 '64 bit Prime OS Operating System',
 '64 bit Windows 10 Operating System',
 '64 bit Windows 11 Home Operating System',
 '64 bit Windows 11 Operating System',
 'Chrome Operating System',
 'DOS Operating System',
 'Mac OS Operating System',
 'Microsoft Office 2021, Windows 11',
 'Microsoft Office 2021, Windows 11 Home',
 'Microsoft Office 2021, Windows 11 Pro',
 'Microsoft Office 2024, Windows 11 Home',
 'Microsoft Office 2024, Windows 11 Pro',
 'Microsoft Office Pro Plus 2021, Windows 11 Home',
 'Microsoft office 2021, Windows 11 Pro',
 'Office Home & Student 2021, Microsoft Windows, 15 Months Macafee Antivirus',
 'Operating System, HP Support Assistant, HP PC Hardware Diagnostics',
 'WINDOWS 11',
 'WINDOWS 11 , OFFICE H&S',
 'WINDOWS 11 HOME, Office H&S 2024',
 'WINDOWS 11, MS OFFICE',
 'Windows',
 'Windows 10 Operating System',
 'Windows 11',
 

In [30]:
conditions = [df['Operating System'].str.contains('Windows|windows'), 
df['Operating System'].str.contains('Mac|mac')             ]

value = ['windows','macos']
clean['os'] = np.select(conditions, value, default = 'linux')

In [31]:
clean['os']

0       windows
1       windows
2       windows
3       windows
4         linux
         ...   
1911    windows
1912    windows
1913    windows
1914    windows
1915    windows
Name: os, Length: 1916, dtype: str

In [32]:
clean

,Price,os
0,47990,windows
1,34990,windows
2,63990,windows
3,119990,windows
4,14990,linux
...,...,...
1911,59390,windows
1912,63990,windows
1913,49990,windows
1914,86755,windows


In [33]:
df[~df['Display'].str.contains('Inch|inch')]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [34]:
clean['display_size'] = df['Display'].str.extract(r'\(([\d.]+)\s*Inch', flags=re.IGNORECASE).astype(float)

In [35]:
clean

,Price,os,display_size
0,47990,windows,14.0
1,34990,windows,14.0
2,63990,windows,15.6
3,119990,windows,15.3
4,14990,linux,11.6
...,...,...,...
1911,59390,windows,14.0
1912,63990,windows,15.6
1913,49990,windows,15.6
1914,86755,windows,15.6


In [36]:
df

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
0,0,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹47,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
1,1,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,"₹34,990",AMD Ryzen 3 Processor,AMD Radeon Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Home Operating System,35.56 cm (14 inch) Display
2,2,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,"₹63,990",Intel Core i5 Processor (13th Gen),Intel Iris Xe (Integrated),16 GB LPDDR4X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
3,3,DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...,"₹1,19,990",AMD Ryzen 7 Processor,NVIDIA GeForce RTX 3050,16 GB DDR5 RAM,512 GB SSD,Windows 11 Home Operating System,38.86 cm (15.3 inch) Display
4,4,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,"₹14,990",MediaTek Kompanio 520 Processor,ARM Mali (Integrated),4 GB LPDDR4X RAM,32 GB EMMC Storage,Chrome Operating System,29.46 cm (11.6 Inch) Display
...,...,...,...,...,...,...,...,...,...
1911,1911,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,"₹59,390",Intel Core i3 Processor (13th Gen),Intel UHD Graphics (Integrated),8 GB DDR5 RAM,512 GB SSD,"Windows 11 Home, Microsoft Office 2024",35.56 cm (14 inch) Display
1912,1912,Samsung Galaxy Book4 Edge Series Copilot AI-PC...,"₹63,990",Snapdragon X Processor,Qualcomm Adreno,16 GB LPDDR5X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
1913,1913,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹49,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 inch) Display
1914,1914,Acer Aspire 7 Intel Core i5 13th Gen - (16 GB/...,"₹86,755",Intel Core i5 Processor (13th Gen),NVIDIA GeForce RTX 2050,16 GB DDR4 RAM,512 GB SSD,Windows 11 Operating System,39.62 cm (15.6 inch) Display


In [37]:
clean['brand'] = df['Title'].str.extract(r'^(\w+)')

In [38]:
set(df["RAM"])

{'12 GB DDR4 RAM',
 '12 GB DDR5 RAM',
 '12 GB LPDDR4X RAM',
 '12 GB LPDDR5 RAM',
 '16 GB DDR4 RAM',
 '16 GB DDR5 RAM',
 '16 GB DDR5X RAM',
 '16 GB LPDDR4X RAM',
 '16 GB LPDDR5 RAM',
 '16 GB LPDDR5X RAM',
 '16 GB Unified Memory RAM',
 '24 GB DDR4 RAM',
 '24 GB DDR5 RAM',
 '24 GB LPDDR5X RAM',
 '24 GB Unified Memory RAM',
 '32 GB DDR4 RAM',
 '32 GB DDR5 RAM',
 '32 GB LPDDR4X RAM',
 '32 GB LPDDR5 RAM',
 '32 GB LPDDR5X RAM',
 '36 GB Unified Memory RAM',
 '4 GB DDR4 RAM',
 '4 GB LPDDR4 RAM',
 '4 GB LPDDR4X RAM',
 '4 GB LPDDR5 RAM',
 '4 GB LPDDR5X RAM',
 '48 GB Unified Memory RAM',
 '64 GB LPDDR5X RAM',
 '8 GB DDR4 RAM',
 '8 GB DDR5 RAM',
 '8 GB LPDDR4 RAM',
 '8 GB LPDDR4X RAM',
 '8 GB LPDDR5 RAM',
 '8 GB LPDDR5X RAM',
 '8 GB SDR RAM'}

In [39]:
clean[['RAM', 'ddr']] = df['RAM'].str.strip().str.extract(r'(\d+)\s*GB\s+(.*?)\s+RAM')
clean['RAM'] = clean['RAM'].astype(int)

In [40]:
clean

,Price,os,display_size,brand,RAM,ddr
0,47990,windows,14.0,ASUS,8,DDR4
1,34990,windows,14.0,Acer,8,DDR4
2,63990,windows,15.6,Samsung,16,LPDDR4X
3,119990,windows,15.3,DELL,16,DDR5
4,14990,linux,11.6,Lenovo,4,LPDDR4X
...,...,...,...,...,...,...
1911,59390,windows,14.0,Lenovo,8,DDR5
1912,63990,windows,15.6,Samsung,16,LPDDR5X
1913,49990,windows,15.6,ASUS,8,DDR4
1914,86755,windows,15.6,Acer,16,DDR4


In [41]:
set(clean['ddr'])

{'DDR4',
 'DDR5',
 'DDR5X',
 'LPDDR4',
 'LPDDR4X',
 'LPDDR5',
 'LPDDR5X',
 'SDR',
 'Unified Memory'}

In [42]:
df['Storage'] = df['Storage'].str.replace('Flash Storage', 'SSD')

In [43]:
clean['is_ssd'] = df['Storage'].str.contains('SSD|ssd').astype(int)
clean['is_hdd'] = df['Storage'].str.contains('HDD|hdd').astype(int)

In [44]:
clean

,Price,os,display_size,brand,RAM,ddr,is_ssd,is_hdd
0,47990,windows,14.0,ASUS,8,DDR4,1,0
1,34990,windows,14.0,Acer,8,DDR4,1,0
2,63990,windows,15.6,Samsung,16,LPDDR4X,1,0
3,119990,windows,15.3,DELL,16,DDR5,1,0
4,14990,linux,11.6,Lenovo,4,LPDDR4X,0,0
...,...,...,...,...,...,...,...,...
1911,59390,windows,14.0,Lenovo,8,DDR5,1,0
1912,63990,windows,15.6,Samsung,16,LPDDR5X,1,0
1913,49990,windows,15.6,ASUS,8,DDR4,1,0
1914,86755,windows,15.6,Acer,16,DDR4,1,0


In [45]:
clean['processor_brand'] = df['Processor'].str.extract(r'^(\w+)')

In [46]:
clean

,Price,os,display_size,brand,RAM,ddr,is_ssd,is_hdd,processor_brand
0,47990,windows,14.0,ASUS,8,DDR4,1,0,Intel
1,34990,windows,14.0,Acer,8,DDR4,1,0,AMD
2,63990,windows,15.6,Samsung,16,LPDDR4X,1,0,Intel
3,119990,windows,15.3,DELL,16,DDR5,1,0,AMD
4,14990,linux,11.6,Lenovo,4,LPDDR4X,0,0,MediaTek
...,...,...,...,...,...,...,...,...,...
1911,59390,windows,14.0,Lenovo,8,DDR5,1,0,Intel
1912,63990,windows,15.6,Samsung,16,LPDDR5X,1,0,Snapdragon
1913,49990,windows,15.6,ASUS,8,DDR4,1,0,Intel
1914,86755,windows,15.6,Acer,16,DDR4,1,0,Intel


In [47]:
set(clean['processor_brand'])

{'AMD', 'Apple', 'Intel', 'MediaTek', 'Qualcomm', 'Snapdragon'}

In [48]:
set(df['Processor'])

{'AMD Athlon Dual Core Processor',
 'AMD Ryzen 3 Dual Core Processor',
 'AMD Ryzen 3 Dual Core Processor (11th Gen)',
 'AMD Ryzen 3 Dual Core Processor (13th Gen)',
 'AMD Ryzen 3 Dual Core Processor (3rd Gen)',
 'AMD Ryzen 3 Dual Core Processor (5th Gen)',
 'AMD Ryzen 3 Dual Core Processor (7th Gen)',
 'AMD Ryzen 3 Octa Core Processor',
 'AMD Ryzen 3 Processor',
 'AMD Ryzen 3 Quad Core Processor',
 'AMD Ryzen 3 Quad Core Processor (5th Gen)',
 'AMD Ryzen 5 Dual Core Processor',
 'AMD Ryzen 5 Hexa Core Processor',
 'AMD Ryzen 5 Hexa Core Processor (7th Gen)',
 'AMD Ryzen 5 Octa Core Processor (7th Gen)',
 'AMD Ryzen 5 Processor',
 'AMD Ryzen 5 Processor (4th Gen)',
 'AMD Ryzen 5 Processor (7th Gen)',
 'AMD Ryzen 5 Quad Core Processor',
 'AMD Ryzen 5 Quad Core Processor (7th Gen)',
 'AMD Ryzen 7 Hexa Core Processor',
 'AMD Ryzen 7 Octa Core Processor',
 'AMD Ryzen 7 Octa Core Processor (7th Gen)',
 'AMD Ryzen 7 Processor',
 'AMD Ryzen 7 Processor (7th Gen)',
 'AMD Ryzen 9 16 Core Process

In [49]:
cpu_scores = {
    'AMD Athlon Dual Core Processor': 3150,
    'AMD Ryzen 3 Dual Core Processor': 4100,
    'AMD Ryzen 3 Dual Core Processor (11th Gen)': 6150,
    'AMD Ryzen 3 Dual Core Processor (13th Gen)': 8500,
    'AMD Ryzen 3 Dual Core Processor (3rd Gen)': 4100,
    'AMD Ryzen 3 Dual Core Processor (5th Gen)': 4850,
    'AMD Ryzen 3 Dual Core Processor (7th Gen)': 8200,
    'AMD Ryzen 3 Octa Core Processor': 10400,
    'AMD Ryzen 3 Processor': 7400,
    'AMD Ryzen 3 Quad Core Processor': 7800,
    'AMD Ryzen 3 Quad Core Processor (5th Gen)': 9300,
    'AMD Ryzen 5 Dual Core Processor': 5400,
    'AMD Ryzen 5 Hexa Core Processor': 13200,
    'AMD Ryzen 5 Hexa Core Processor (7th Gen)': 16400,
    'AMD Ryzen 5 Octa Core Processor (7th Gen)': 19500,
    'AMD Ryzen 5 Processor': 13100,
    'AMD Ryzen 5 Processor (4th Gen)': 11200,
    'AMD Ryzen 5 Processor (7th Gen)': 16000,
    'AMD Ryzen 5 Quad Core Processor': 8100,
    'AMD Ryzen 5 Quad Core Processor (7th Gen)': 9600,
    'AMD Ryzen 7 Hexa Core Processor': 14600,
    'AMD Ryzen 7 Octa Core Processor': 20443,
    'AMD Ryzen 7 Octa Core Processor (7th Gen)': 23800,
    'AMD Ryzen 7 Processor': 18500,
    'AMD Ryzen 7 Processor (7th Gen)': 22800,
    'AMD Ryzen 9 16 Core Processor': 53764,
    'AMD Ryzen 9 16 Core Processor (8th Gen)': 54500,
    'AMD Ryzen 9 Octa Core Processor': 24600,
    'AMD Ryzen AI 5 Hexa Core Processor (7th Gen)': 19400,
    'AMD Ryzen AI 5 Quad Core Processor': 14800,
    'AMD Ryzen AI Max Processor': 46000,
    'Apple M2 Processor': 15646,
    'Apple M5 Max Processor': 52000,
    'Apple M5 Pro Processor': 39200,
    'Apple M5 Processor': 26735,
    'Intel Celeron Dual Core Processor': 2250,
    'Intel Celeron Quad Core Processor': 4100,
    'Intel Core 3 Processor': 11400,
    'Intel Core 3 Processor (14th Gen)': 11800,
    'Intel Core 5 Processor': 17400,
    'Intel Core 5 Processor (13th Gen)': 17200,
    'Intel Core 5 Processor (14th Gen)': 18200,
    'Intel Core 5 Processor (Series 2)': 18600,
    'Intel Core 5 Processor (Series 3)': 20500,
    'Intel Core 7 Processor': 21200,
    'Intel Core 7 Processor (14th Gen)': 22500,
    'Intel Core Ultra 5 Processor': 19200,
    'Intel Core Ultra 5 Processor (14th Gen)': 18900,
    'Intel Core Ultra 5 Processor (1st Gen)': 18900,
    'Intel Core Ultra 5 Processor (Series 2)': 20600,
    'Intel Core Ultra 5 Processor (Series 3)': 23200,
    'Intel Core Ultra 7 Processor': 23800,
    'Intel Core Ultra 7 Processor (Series 3)': 27800,
    'Intel Core Ultra 9 Processor': 28800,
    'Intel Core Ultra X7 Processor': 26500,
    'Intel Core i3 Processor': 9400,
    'Intel Core i3 Processor (10th Gen)': 4150,
    'Intel Core i3 Processor (11th Gen)': 6150,
    'Intel Core i3 Processor (12th Gen)': 10400,
    'Intel Core i3 Processor (13th Gen)': 11500,
    'Intel Core i3 Processor (14th Gen)': 12400,
    'Intel Core i5 Processor (11th Gen)': 9286,
    'Intel Core i5 Processor (12th Gen)': 15400,
    'Intel Core i5 Processor (13th Gen)': 17600,
    'Intel Core i5 Processor (14th Gen)': 19600,
    'Intel Core i5 Processor (5th Gen)': 2650,
    'Intel Core i5 Processor (8th Gen)': 6150,
    'Intel Core i7 Processor (11th Gen)': 19535,
    'Intel Core i7 Processor (12th Gen)': 24925,
    'Intel Core i7 Processor (13th Gen)': 26400,
    'Intel Core i7 Processor (14th Gen)': 28500,
    'Intel Core i9 Processor (13th Gen)': 31200,
    'Intel N50 Processor': 3200,
    'Intel Pentium Quad Core Processor': 4800,
    'Intel Pentium Silver Processor': 3600,
    'Intel Ryzen 7 Octa Core Processor (6th Gen)': 21500,
    'MediaTek Genio 720 (MT8391) Processor': 6800,
    'MediaTek Helio G99 (MT8781) Processor': 4950,
    'MediaTek Kompanio 520 Processor': 2650,
    'MediaTek MT8183 Processor': 1850,
    'MediaTek MediaTek Kompanio 528 Processor': 2800,
    'MediaTek MediaTek Kompanio 540 Processor': 3250,
    'MediaTek MediaTek Kompanio 838 Processor': 5600,
    'Qualcomm Snapdragon X Processor': 23400,
    'Snapdragon X Plus Processor': 19600,
    'Snapdragon X Processor': 23400,
    'Snapdragon X2 Elite Processor': 31500,
    'Snapdragon X2 Plus Processor': 24500
}

In [50]:
clean['cpu_power']= df['Processor'].map(cpu_scores)

In [51]:
clean['Title'] = df['Title']

In [52]:
clean[clean.isna().any(axis=1)]

,Price,os,display_size,brand,RAM,ddr,is_ssd,is_hdd,processor_brand,cpu_power,Title


In [53]:
df

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
0,0,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹47,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
1,1,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,"₹34,990",AMD Ryzen 3 Processor,AMD Radeon Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Home Operating System,35.56 cm (14 inch) Display
2,2,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,"₹63,990",Intel Core i5 Processor (13th Gen),Intel Iris Xe (Integrated),16 GB LPDDR4X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
3,3,DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...,"₹1,19,990",AMD Ryzen 7 Processor,NVIDIA GeForce RTX 3050,16 GB DDR5 RAM,512 GB SSD,Windows 11 Home Operating System,38.86 cm (15.3 inch) Display
4,4,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,"₹14,990",MediaTek Kompanio 520 Processor,ARM Mali (Integrated),4 GB LPDDR4X RAM,32 GB EMMC Storage,Chrome Operating System,29.46 cm (11.6 Inch) Display
...,...,...,...,...,...,...,...,...,...
1911,1911,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,"₹59,390",Intel Core i3 Processor (13th Gen),Intel UHD Graphics (Integrated),8 GB DDR5 RAM,512 GB SSD,"Windows 11 Home, Microsoft Office 2024",35.56 cm (14 inch) Display
1912,1912,Samsung Galaxy Book4 Edge Series Copilot AI-PC...,"₹63,990",Snapdragon X Processor,Qualcomm Adreno,16 GB LPDDR5X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
1913,1913,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹49,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 inch) Display
1914,1914,Acer Aspire 7 Intel Core i5 13th Gen - (16 GB/...,"₹86,755",Intel Core i5 Processor (13th Gen),NVIDIA GeForce RTX 2050,16 GB DDR4 RAM,512 GB SSD,Windows 11 Operating System,39.62 cm (15.6 inch) Display


In [54]:
df.to_csv("Dirty.csv")

In [55]:
df[df['GPU'].str.contains('ntegrated')]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
0,0,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹47,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
1,1,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,"₹34,990",AMD Ryzen 3 Processor,AMD Radeon Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Home Operating System,35.56 cm (14 inch) Display
2,2,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,"₹63,990",Intel Core i5 Processor (13th Gen),Intel Iris Xe (Integrated),16 GB LPDDR4X RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 Inch) Display
4,4,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,"₹14,990",MediaTek Kompanio 520 Processor,ARM Mali (Integrated),4 GB LPDDR4X RAM,32 GB EMMC Storage,Chrome Operating System,29.46 cm (11.6 Inch) Display
5,5,MOTOROLA Motobook 60 Full Metal WUXGA OLED (i5...,"₹64,990",Intel Core 5 Processor,Intel Iris Xe (Integrated),16 GB DDR5 RAM,512 GB SSD,64 bit Windows 11 Operating System,35.56 cm (14 inch) Display
...,...,...,...,...,...,...,...,...,...
1907,1907,HP 15 Backlit (2026) (i3 14th Gen) Intel Core ...,"₹54,490",Intel Core 3 Processor (14th Gen),Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,"Microsoft Office 2024, Windows 11 Home",39.62 cm (15.6 inch) Display
1909,1909,ASUS ExpertBook P5 with Full Metal OLED AI PC ...,"₹90,990",Intel Core Ultra 5 Processor,Intel Arc Graphics (Integrated),16 GB DDR5 RAM,512 GB SSD,Windows 11 Home Operating System,35.56 cm (14 inch) Display
1911,1911,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,"₹59,390",Intel Core i3 Processor (13th Gen),Intel UHD Graphics (Integrated),8 GB DDR5 RAM,512 GB SSD,"Windows 11 Home, Microsoft Office 2024",35.56 cm (14 inch) Display
1913,1913,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,"₹49,990",Intel Core 3 Processor,Intel UHD Graphics (Integrated),8 GB DDR4 RAM,512 GB SSD,64 bit Windows 11 Operating System,39.62 cm (15.6 inch) Display


In [56]:
row_gpu_fixes = {
    85: 'NVIDIA GeForce RTX 4060',             # Acer Nitro V 16S (Core 5 210H, 8GB dGPU @ ₹1.25L = RTX 4060 / 5060)
    109: 'NVIDIA GeForce RTX 2050',            # Lenovo IdeaPad i5-12450H (4GB dGPU @ ₹79.9k)
    381: 'NVIDIA GeForce RTX 3050',            # ASUS TUF Gaming F16 Core 5 (4GB dGPU @ ₹1.04L)
    446: 'NVIDIA GeForce RTX 3050',            # ASUS TUF Gaming F16 Core 5 (duplicate)
    671: 'NVIDIA GeForce MX450',               # Dell Inspiron i5-11300H (2GB dGPU)
    735: 'NVIDIA GeForce RTX 4050',            # Lenovo LOQ/IdeaPad Pro Ryzen 5 (6GB dGPU @ ₹1.12L)
    895: 'NVIDIA GeForce RTX 3050',            # ASUS TUF A15 Ryzen 7 7445HS (4GB dGPU)
    920: 'Intel UHD Graphics (Integrated)',    # Dell Inspiron 15 100U (₹58k i3 laptop - seller wrote RAM as Graphics)
    925: 'NVIDIA GeForce RTX 4070',            # HP Omen Transcend 16 Ryzen 9 8940HX (8GB dGPU @ ₹2.89L)
    936: 'Intel UHD Graphics (Integrated)',    # Dell DC 15250 i3 14th Gen (₹55k office laptop - integrated)
    965: 'NVIDIA GeForce RTX 4050',            # ASUS ROG/Creator i5 13th Gen (6GB dGPU)
    975: 'NVIDIA GeForce RTX 3050',            # ASUS Vivobook 16X / TUF i5 12th Gen (4GB dGPU)
    1300: 'NVIDIA GeForce MX450',              # Lenovo IdeaPad Slim 5i i5-11300H (2GB dGPU)
    1340: 'NVIDIA GeForce RTX 3050',           # ASUS TUF Gaming F16 Core 5 (duplicate)
    1694: 'NVIDIA GeForce RTX 4050',           # Lenovo Ryzen 5 6GB dGPU (duplicate of 735)
    1820: 'NVIDIA GeForce RTX 3050',           # ASUS Ryzen 7 7445HS 4GB dGPU (duplicate of 895)
    1887: 'Intel UHD Graphics (Integrated)',   # Dell DC 15250 i3 14th Gen (duplicate of 936)
    1910: 'Qualcomm Adreno'                    # HP OmniBook Snapdragon X (seller put 16GB shared RAM as Graphics)
}

# Apply directly to your DataFrame:
df.loc[row_gpu_fixes.keys(), 'GPU'] = pd.Series(row_gpu_fixes)

In [57]:
clean['dedicated_gpu'] = (~df['GPU'].str.contains('ntegrated')).astype(int)

In [58]:
set(df['GPU'])

{'AMD Radeon Graphics (Integrated)',
 'AMD Radeon RX 6700S',
 'ARM Mali (Integrated)',
 'Apple Integrated GPU',
 'Intel Arc Graphics (Integrated)',
 'Intel Iris Xe (Integrated)',
 'Intel UHD Graphics (Integrated)',
 'NVIDIA GeForce GTX 1050 Ti',
 'NVIDIA GeForce GTX 1650',
 'NVIDIA GeForce MX450',
 'NVIDIA GeForce RTX 2050',
 'NVIDIA GeForce RTX 3050',
 'NVIDIA GeForce RTX 3050 Ti',
 'NVIDIA GeForce RTX 3060',
 'NVIDIA GeForce RTX 4050',
 'NVIDIA GeForce RTX 4060',
 'NVIDIA GeForce RTX 4070',
 'NVIDIA GeForce RTX 4080',
 'NVIDIA GeForce RTX 5050',
 'NVIDIA GeForce RTX 5060',
 'NVIDIA GeForce RTX 5070',
 'NVIDIA GeForce RTX 5070 Ti',
 'NVIDIA GeForce RTX 5080',
 'Qualcomm Adreno'}

In [59]:
gpu_scores = {
    'AMD Radeon Graphics (Integrated)': 2700,
    'AMD Radeon RX 6700S': 14972,
    'ARM Mali (Integrated)': 850,
    'Apple Integrated GPU': 8500,
    'Dedicated GPU (16 GB)': 26500,
    'Dedicated GPU (2 GB)': 3800,
    'Dedicated GPU (4 GB)': 8200,
    'Dedicated GPU (6 GB)': 13100,
    'Dedicated GPU (8 GB)': 17500,
    'Intel Arc Graphics (Integrated)': 4800,
    'Intel Iris Xe (Integrated)': 2588,
    'Intel UHD Graphics (Integrated)': 1450,
    'NVIDIA GeForce GTX 1050 Ti': 5918,
    'NVIDIA GeForce GTX 1650': 6968,
    'NVIDIA GeForce RTX 2050': 7540,
    'NVIDIA GeForce RTX 3050': 10119,
    'NVIDIA GeForce RTX 3050 Ti': 10057,
    'NVIDIA GeForce RTX 3060': 13092,
    'NVIDIA GeForce RTX 4050': 14196,
    'NVIDIA GeForce MX450': 3890,
    'NVIDIA GeForce RTX 4060': 17510,
    'NVIDIA GeForce RTX 4070': 19538,
    'NVIDIA GeForce RTX 4080': 24766,
    'NVIDIA GeForce RTX 5050': 14013,
    'NVIDIA GeForce RTX 5060': 16759,
    'NVIDIA GeForce RTX 5070': 19142,
    'NVIDIA GeForce RTX 5070 Ti': 22497,
    'NVIDIA GeForce RTX 5080': 26299,
    'Qualcomm Adreno': 2800
}

In [60]:
clean['gpu_scores'] = df['GPU'].map(gpu_scores)

In [61]:
df[clean.isna().any(axis=1)]

,Unnamed: 0,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display


In [62]:
set(df['GPU'])

{'AMD Radeon Graphics (Integrated)',
 'AMD Radeon RX 6700S',
 'ARM Mali (Integrated)',
 'Apple Integrated GPU',
 'Intel Arc Graphics (Integrated)',
 'Intel Iris Xe (Integrated)',
 'Intel UHD Graphics (Integrated)',
 'NVIDIA GeForce GTX 1050 Ti',
 'NVIDIA GeForce GTX 1650',
 'NVIDIA GeForce MX450',
 'NVIDIA GeForce RTX 2050',
 'NVIDIA GeForce RTX 3050',
 'NVIDIA GeForce RTX 3050 Ti',
 'NVIDIA GeForce RTX 3060',
 'NVIDIA GeForce RTX 4050',
 'NVIDIA GeForce RTX 4060',
 'NVIDIA GeForce RTX 4070',
 'NVIDIA GeForce RTX 4080',
 'NVIDIA GeForce RTX 5050',
 'NVIDIA GeForce RTX 5060',
 'NVIDIA GeForce RTX 5070',
 'NVIDIA GeForce RTX 5070 Ti',
 'NVIDIA GeForce RTX 5080',
 'Qualcomm Adreno'}

In [63]:
clean

,Price,os,display_size,brand,RAM,ddr,is_ssd,is_hdd,processor_brand,cpu_power,Title,dedicated_gpu,gpu_scores
0,47990,windows,14.0,ASUS,8,DDR4,1,0,Intel,11400,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,0,1450
1,34990,windows,14.0,Acer,8,DDR4,1,0,AMD,7400,Acer Aspire 14 AMD Ryzen 3 3200U - (8 GB/512 G...,0,2700
2,63990,windows,15.6,Samsung,16,LPDDR4X,1,0,Intel,17600,Samsung Galaxy Book4 Metal Intel Core i5 13th ...,0,2588
3,119990,windows,15.3,DELL,16,DDR5,1,0,AMD,18500,DELL Alienware 15 AMD Ryzen 7 260 - (16 GB/512...,1,10119
4,14990,linux,11.6,Lenovo,4,LPDDR4X,0,0,MediaTek,2650,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,0,850
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1911,59390,windows,14.0,Lenovo,8,DDR5,1,0,Intel,11500,Lenovo V 14 (2026) DDR5 5600Mhz Intel Core i3 ...,0,1450
1912,63990,windows,15.6,Samsung,16,LPDDR5X,1,0,Snapdragon,23400,Samsung Galaxy Book4 Edge Series Copilot AI-PC...,1,2800
1913,49990,windows,15.6,ASUS,8,DDR4,1,0,Intel,11400,ASUS ExpertBook P1 Intel Core 3 100U - (8 GB/5...,0,1450
1914,86755,windows,15.6,Acer,16,DDR4,1,0,Intel,17600,Acer Aspire 7 Intel Core i5 13th Gen - (16 GB/...,1,7540


In [64]:
clean.insert(0, 'Title', clean.pop('Title') )

In [65]:
ddr_scores = {
    'SDR': 32,              # DDR4-3200 (seller typo for DDR4 SDRAM)
    'DDR4': 32,             # 3200 MT/s
    'LPDDR4': 37,           # 3733 MT/s
    'LPDDR4X': 42,          # 4266 MT/s
    'DDR5': 52,             # 4800–5600 MT/s
    'LPDDR5': 64,           # 6400 MT/s
    'Unified Memory': 70,   # Average of M1/M2/M3 (LPDDR5-6400) & M4/M5 (LPDDR5X-7500+)
    'DDR5X': 75,            # 7500 MT/s
    'LPDDR5X': 75           # 7500–8533 MT/s
}

In [66]:
clean['ddr'] = clean['ddr'].map(ddr_scores)

In [67]:
clean[clean['Title'].str.contains('LOQ')]

,Title,Price,os,display_size,brand,RAM,ddr,is_ssd,is_hdd,processor_brand,cpu_power,dedicated_gpu,gpu_scores
158,Lenovo LOQ 2025 AMD Ryzen 7 Octa Core 250 - (1...,165990,windows,15.6,Lenovo,16,52,1,0,AMD,20443,1,14013
213,Lenovo LOQ Essential AMD Ryzen 7 Octa Core 773...,130990,windows,15.6,Lenovo,16,52,1,0,AMD,20443,1,14196
287,Lenovo LOQ Essential AMD Ryzen 7 Octa Core 773...,130990,windows,15.6,Lenovo,16,52,1,0,AMD,20443,1,14196
530,Lenovo LOQ 15IRX9. Intel Core i5 13th Gen 1345...,129990,windows,15.6,Lenovo,16,52,1,0,Intel,17600,1,14196
600,Lenovo LOQ Intel Core i7 14th Gen 14650HX - (1...,179990,windows,15.6,Lenovo,16,52,1,0,Intel,28500,1,14196
619,Lenovo LOQ 15IAX9 Intel Core i5 12th Gen 12450...,114990,windows,15.6,Lenovo,16,52,1,0,Intel,15400,1,10119
786,Lenovo LOQ Intel Core i7 12th Gen 12650HX - (1...,139471,windows,15.6,Lenovo,16,52,1,0,Intel,24925,1,14196
1069,Lenovo LOQ 2025 Intel Core i7 14th Gen 14700HX...,159990,windows,15.6,Lenovo,16,52,1,0,Intel,28500,1,14013
1101,Lenovo LOQ 2025 Intel Core i7 14th Gen 14700HX...,186990,windows,15.6,Lenovo,16,52,1,0,Intel,28500,1,16759
1185,Lenovo LOQ 2025 AMD Ryzen 7 Octa Core 250 - (1...,165990,windows,15.6,Lenovo,16,52,1,0,AMD,20443,1,14013


In [68]:
clean.to_csv("clean_data.csv")

In [367]:
set(clean['ddr'])

{32, 37, 42, 52, 64, 70, 75}

In [368]:
set(df['GPU'])

{'AMD Radeon Graphics (Integrated)',
 'AMD Radeon RX 6700S',
 'ARM Mali (Integrated)',
 'Apple Integrated GPU',
 'Intel Arc Graphics (Integrated)',
 'Intel Iris Xe (Integrated)',
 'Intel UHD Graphics (Integrated)',
 'NVIDIA GeForce GTX 1050 Ti',
 'NVIDIA GeForce GTX 1650',
 'NVIDIA GeForce MX450',
 'NVIDIA GeForce RTX 2050',
 'NVIDIA GeForce RTX 3050',
 'NVIDIA GeForce RTX 3050 Ti',
 'NVIDIA GeForce RTX 3060',
 'NVIDIA GeForce RTX 4050',
 'NVIDIA GeForce RTX 4060',
 'NVIDIA GeForce RTX 4070',
 'NVIDIA GeForce RTX 4080',
 'NVIDIA GeForce RTX 5050',
 'NVIDIA GeForce RTX 5060',
 'NVIDIA GeForce RTX 5070',
 'NVIDIA GeForce RTX 5070 Ti',
 'NVIDIA GeForce RTX 5080',
 'Qualcomm Adreno'}

In [369]:
df[df['GPU'].str.contains('Dedicated')]

,Title,Price,Processor,GPU,RAM,Storage,Operating System,Display
